---
image: example.gif
pub-info:
    abstract: |
        A packing robot model comparing two ways of moving entities between steps: vidigi's default
        interpolated pathing, versus precalculated paths for when the interpolated route would cut
        across a background image awkwardly. Useful once your layout gets complex enough that a
        straight line between steps stops looking right.
execute: 
  enabled: true
---

# Precalculated pathing

In [ ]:
import os
from vidigi.utils import EventPosition, create_event_position_df
from vidigi.prep import reshape_for_animations, generate_animation_df
from vidigi.animation import generate_animation, animate_activity_log
import pandas as pd
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
#| echo: false
#| output: asis
# Path to the external Python script
file_path = "warehouse_robot.py"

# Read the file content
if os.path.exists(file_path):
    with open(file_path, "r") as f:
        code_content = f.read()
else:
    code_content = "File not found."
with open(file_path, "r") as f:
    code_content = f.read()

# Print the Quarto `{details}` block for collapsible output
print(f"""
:::{{.callout-note collapse="true"}}
### View Imported Code, which has had logging steps added at the appropriate points in the 'model' class

```python
{code_content}
```

:::

""")

In [ ]:
# Define positions for animation
event_positions = create_event_position_df([
    EventPosition(event='arrival', x=0, y=550, label="Entrance"),

    EventPosition(event='pickup_1', x=40, y=500, label="Pickup 1"),
    EventPosition(event='pickup_2', x=170, y=500, label="Pickup 2"),
    EventPosition(event='pickup_3', x=350, y=500, label="Pickup 3"),
    EventPosition(event='pickup_4', x=500, y=500, label="Pickup 4"),
    EventPosition(event='pickup_5', x=40, y=60, label="Pickup 5"),
    EventPosition(event='pickup_6', x=170, y=60, label="Pickup 6"),
    EventPosition(event='pickup_7', x=350, y=60, label="Pickup 7"),
    EventPosition(event='pickup_8', x=500, y=60, label="Pickup 8"),

    EventPosition(event='packing', x=300, y=300, label="Packing"),
    EventPosition(event='maintenance', x=600, y=300, label="Maintenance"),

    EventPosition(event='depart', x=650, y=50, label="Exit")
])

In [ ]:
event_log_df = pd.read_csv("robot_log.csv")
event_log_df.head()

In [ ]:
STEP_SNAPSHOT_MAX = 999
LIMIT_DURATION = int(max(event_log_df[event_log_df["event_type"]!="position_poll"]['time']))
WRAP_QUEUES_AT = 999

In [ ]:
event_log_df_filtered = event_log_df[~event_log_df["event_type"].isin(["position_poll", "action"])][['entity_id', 'event_type', 'event', 'time', 'pathway']]
event_log_df_filtered.head(10)

We'll first run this while allowing vidigi to handle the pathing. This means that the path between each step will be *interpolated*. 

In [ ]:
animate_activity_log(
    event_log=event_log_df_filtered,
    event_position_df=event_positions,
    wrap_queues_at=WRAP_QUEUES_AT,
    limit_duration=LIMIT_DURATION,
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    every_x_time_units=1,
    debug_mode=True,
    custom_entity_icon_list=["🤖"]
)

If we were to put in a background illustrating the paths the robots should follow, this will look bad: 

In [ ]:
animate_activity_log(
    event_log=event_log_df_filtered,
    event_position_df=event_positions,
    wrap_queues_at=WRAP_QUEUES_AT,
    limit_duration=LIMIT_DURATION,
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    every_x_time_units=1,
    debug_mode=True,
    display_stage_labels=False,
    custom_entity_icon_list=["🤖"],
    add_background_image="https://raw.githubusercontent.com/hsma-tools/vidigi/refs/heads/main/examples/example_16_packing_robot/warehouse.png",
    background_image_opacity=1, # New parameter in 1.1.0
    override_x_max=650,
    override_y_max=550,
    plotly_width=1300,
    plotly_height=800,
)

However, in this particular model, we have been polling the location of the robot after every step. 

This might be imporant to visualise accurately in certain models - for example, to demonstrate why a robot might have to wait for another robot to move out of the way, and to assure stakeholders that such a thing has been recorded accurately. 

Let's see how we can combine vidigi with these polled locations. 

In [ ]:
full_entity_df = reshape_for_animations(
    event_log=event_log_df[~event_log_df["event_type"].isin(["position_poll", "action"])][['entity_id', 'event_type', 'event', 'time', 'pathway']],
    limit_duration=LIMIT_DURATION,
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    every_x_time_units=1,
    debug_mode=True
    )

full_entity_df

In [ ]:
full_entity_df_plus_pos = generate_animation_df(
    full_entity_df=full_entity_df,
    event_position_df=event_positions,
    wrap_queues_at=WRAP_QUEUES_AT,
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    debug_mode=True,
    custom_entity_icon_list=["🤖"]
    )

full_entity_df_plus_pos

Now we can replace our calculated x_final and y_final coordinates with the values from the polling. 

First, we pull this data out of our event logs and rename the columns to match the relevant columns in our transformed event logging dataset. 

In [ ]:
entity_position_df = event_log_df[event_log_df["event_type"]=="position_poll"][['entity_id', 'time', 'x', 'y']].reset_index(drop=True)
entity_position_df = entity_position_df.rename(columns={"x": "x_final", "y": "y_final", "time": "snapshot_time"})
entity_position_df

We then drop our original x_final and y_final columns from our transformed dataset, replacing them with those from the polling in our model. 

In [ ]:
full_entity_df_plus_pos_manual_locations = (
    full_entity_df_plus_pos
    .drop(columns=["x_final", "y_final"])
    .merge(entity_position_df, on=["entity_id", "snapshot_time"])
)

full_entity_df_plus_pos_manual_locations

Finally, we then generate our animation as usual - making sure to refer to our updated dataframe. 

In [ ]:
fig = generate_animation(
        full_entity_df_plus_pos=full_entity_df_plus_pos_manual_locations.sort_values(['entity_id', 'snapshot_time']),
        event_position_df= event_positions,
        simulation_time_unit="seconds",
        display_stage_labels=False,
        setup_mode=False,
        start_time="07:00:00",
        time_display_units="%H:%M:%S",
        debug_mode=True,
        add_background_image="https://raw.githubusercontent.com/hsma-tools/vidigi/refs/heads/main/examples/example_16_packing_robot/warehouse.png",
        background_image_opacity=1, # New parameter in 1.1.0
        override_x_max=650,
        override_y_max=550,
        plotly_width=1300,
        plotly_height=800,
        entity_icon_size=50,
        frame_duration=200,
        frame_transition_duration=300

    )

fig

# Multiple Packers

Let's finally repeat this with an instance with multiple packers. 

In this very simplistic model, blocking of corridors is not implemented - but is the kind of thing that would visualise well with this kind of pre-calculated movement. 

In [ ]:
event_log_df_multiple = pd.read_csv("robot_log_multiple.csv")

full_entity_df_multiple = reshape_for_animations(
    event_log=event_log_df_multiple[~event_log_df_multiple["event_type"].isin(["position_poll", "action"])][['entity_id', 'event_type', 'event', 'time', 'pathway']],
    limit_duration=LIMIT_DURATION,
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    every_x_time_units=1,
    debug_mode=True
    )

full_entity_df_plus_pos_multiple = generate_animation_df(
    full_entity_df=full_entity_df_multiple,
    event_position_df=event_positions,
    wrap_queues_at=WRAP_QUEUES_AT,
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    debug_mode=True,
    custom_entity_icon_list=["🤖"]
    # custom_entity_icon_list=["🟦", "🟫", "🟪", "🟧", "🟥", "🟨", "🟩", "◻️"]
    )

full_entity_df_plus_pos_multiple

In [ ]:
entity_position_df_multiple = (
    event_log_df_multiple[event_log_df_multiple["event_type"]=="position_poll"]
    [['entity_id', 'time', 'x', 'y']]
    .reset_index(drop=True)
    .rename(columns={"x": "x_final", "y": "y_final", "time": "snapshot_time"})
    )

# entity_position_df_multiple["snapshot_time"] = entity_position_df_multiple["snapshot_time"].astype('int')

entity_position_df_multiple

In [ ]:
full_entity_df_plus_pos_manual_locations_multiple = (
    full_entity_df_plus_pos_multiple
    .drop(columns=["x_final", "y_final"])
    .merge(entity_position_df_multiple, on=["entity_id", "snapshot_time"])
)

full_entity_df_plus_pos_manual_locations_multiple

In [ ]:
fig_multiple = generate_animation(
        full_entity_df_plus_pos=full_entity_df_plus_pos_manual_locations_multiple.sort_values(['entity_id', 'snapshot_time']),
        event_position_df= event_positions,
        simulation_time_unit="seconds",
        display_stage_labels=False,
        setup_mode=False,
        start_time="07:00:00",
        time_display_units="%H:%M:%S",
        debug_mode=True,
        add_background_image="https://raw.githubusercontent.com/hsma-tools/vidigi/refs/heads/main/examples/example_16_packing_robot/warehouse.png",
        background_image_opacity=1, # New parameter in 1.1.0
        override_x_max=650,
        override_y_max=550,
        plotly_width=1300,
        plotly_height=800,
        entity_icon_size=30

    )

fig_multiple